# Phase 8 — Guardrails

**Goal**: An agent's answer can be *wrong* in ways the agent itself won't notice — an empty reply, a made-up customer segment ("Platinum"), an absurd number. **Guardrails** are checks that run on the answer *before the user ever sees it*, and can flag or block bad output.

Two layers, catching two different kinds of mistake:

| Layer | Catches | How |
|---|---|---|
| **Deterministic** | *structural* errors — empty answer, invented segment label, absurd number | plain Python rules. Fast, free, exact. |
| **LLM-judge** | *semantic* errors — "this doesn't actually answer the question" | a second LLM scores the answer. |

You need both: rules can't judge meaning, and an LLM judge is fuzzy about structure. (This is the trade-off noted in the project README.)

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Deterministic guardrail** | A code rule that inspects the answer text. Returns a list of violations. |
| **Hallucinated label** | A plausible-sounding value the agent invented — e.g. a "Platinum" segment that doesn't exist. |
| **LLM-judge** | An LLM acting as a reviewer — like the Phase 2 critic, reused here as a guardrail. |
| **Defence in depth** | Multiple independent checks, so a miss by one is caught by another. |

## Acceptance criteria
1. The deterministic guardrails **catch** a deliberately bad answer
2. A real, clean agent answer **passes** both guardrail layers
3. `Trace` exported to `traces/traces.jsonl`

## 1. Setup

In [ ]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

In [ ]:
import pandas as pd

from orchestrator.tools import get_retail_df
from orchestrator import tools
from orchestrator.observability import Trace, Timer
from orchestrator.agents import run_agent, parse_verdict
# Phase 8 adds the guardrails module:
from orchestrator.guardrails import run_deterministic_guardrails, GuardrailResult
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")

df = get_retail_df()
print(f"Loaded {len(df):,} rows")
print(f"Dev model: {DEV_MODEL}")

## 2. Ground truth

A normal question for the real-answer test: top 5 countries by revenue in 2011.

In [ ]:
df_2011 = df[(df["Quantity"] > 0) & (df["InvoiceDate"].dt.year == 2011)]

ground_truth = (
    df_2011.groupby("Country")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
print("Top 5 countries by revenue, 2011:")
print(ground_truth)

expected_countries = sorted(ground_truth["Country"])
print()
print("Expected countries in the answer:", expected_countries)

## 3. Rebuild the retail tool + DataAgent (recap)

Same `query_retail` tool and DataAgent. Just run it.

In [ ]:
@tool(
    "query_retail",
    description=(
        "Query the retail transactions dataset to return the top N entries "
        "ranked by a metric. Arguments: year (optional), country (optional - "
        "OMIT to include all countries), top_n (default 10), group_by (one of "
        "'StockCode', 'Country', 'Customer ID'), metric (one of 'revenue', "
        "'Quantity'). Returns a list of dicts, one row each."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "year":     {"type": "integer", "description": "Calendar year filter, e.g. 2011"},
            "country":  {"type": "string",  "description": "Optional country filter. OMIT to include all countries."},
            "top_n":    {"type": "integer", "description": "How many rows to return"},
            "group_by": {"type": "string",  "description": "'StockCode', 'Country', or 'Customer ID'"},
            "metric":   {"type": "string",  "description": "'revenue' or 'Quantity'"},
        },
        "required": [],
    },
)
async def query_retail_tool(args):
    """Adapter: the agent passes args as a dict; we call the pandas function."""
    rows = tools.query_retail(
        year=args.get("year"),
        country=args.get("country"),
        top_n=args.get("top_n", 10),
        group_by=args.get("group_by", "StockCode"),
        metric=args.get("metric", "revenue"),
    )
    import json
    return {"content": [{"type": "text", "text": json.dumps(rows, default=str)}]}


retail_server = create_sdk_mcp_server(name="retail", version="1.0.0", tools=[query_retail_tool])

data_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=(
        "You are a retail data analyst. Answer using the query_retail tool "
        "and present the result as a markdown table. Report ONLY the figures "
        "the tool returns - do NOT add your own percentages, calculations, "
        "or extra commentary, and never invent numbers."
    ),
    mcp_servers={"retail": retail_server},
    allowed_tools=["mcp__retail__query_retail"],
    max_turns=5,
)
print("[OK] Retail tool + DataAgent ready")

## 4. Deterministic guardrails — catching a bad answer

`run_deterministic_guardrails()` (in `orchestrator/guardrails.py`) runs a set of plain Python rules: not-empty, no hallucinated segment names, no absurd numbers. It returns a `GuardrailResult` with `passed` and a list of `violations`.

Run the cell below — we feed it a deliberately **bad** answer and watch the rules catch it.

In [ ]:
bad_answer = (
    "Our Platinum and Gold customers were the top performers, "
    "generating 9,999,999,999 in revenue last quarter."
)

result = run_deterministic_guardrails(bad_answer)
print("Bad answer:")
print(f"  {bad_answer}")
print()
print(f"Guardrails passed: {result.passed}")
print("Violations caught:")
for v in result.violations:
    print(f"  - {v}")

## 5. Add your own deterministic check

Guardrails are just functions: take the answer text, return a list of violation strings (empty list = no problem). Here you write one more.

**TODO 1**: write `check_has_numbers` — a data answer should contain at least one digit; flag it if it doesn't.

In [ ]:
# ----- TODO 1: write the check_has_numbers guardrail -----------------
def check_has_numbers(answer):
    for ch in answer:
        if ch.isdigit():
            return []
    return ["answer contains no numbers"]
# ---------------------------------------------------------------------

# quick self-test
print("has numbers ->", check_has_numbers("Revenue was 8,254,828"))   # expect []
print("no numbers  ->", check_has_numbers("Revenue was very high"))    # expect a violation

## 6. The LLM-judge guardrail

Rules can't tell whether an answer is *evasive* or *off-topic* — that needs judgement. The **LLM-judge** is a second LLM that reviews the answer. It's the Phase 2 critic pattern, reused as a guardrail: it replies `APPROVE` / `REJECT`, and `parse_verdict()` reads it.

**TODO 2**: write the LLM-judge's system prompt.

In [ ]:
judge_system_prompt = (
    # ----- TODO 2: write the LLM-judge's system prompt -------------------
    "You are an answer-quality judge for a customer analytics assistant. "
    "You will be shown a QUESTION and an ANSWER, and you must decide "
    "whether the answer is acceptable to show the user. "
    "Reply with a single bare word — APPROVE or REJECT — as the very first "
    "word of your reply, with no quotes or punctuation before it. If you "
    "REJECT, follow that word with one sentence explaining what is wrong. "
    "Approve when the answer directly addresses the question, is not "
    "evasive or empty, and contains no self-contradictory or clearly "
    "implausible claims. Reject when the answer dodges the question, "
    "ignores part of it, or makes a claim that obviously cannot be true. "
    "Do NOT require source citations — the numbers come from a trusted "
    "internal data tool, so an uncited table is fine."
    # ---------------------------------------------------------------------
)

judge_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=judge_system_prompt,
    max_turns=2,
)
print("[OK] LLM-judge configured")

## 7. Both layers together

`run_all_guardrails()` runs everything: the deterministic checks, your `check_has_numbers`, and the LLM judge. If *any* layer flags a violation, the answer fails. No TODO here — just run it.

In [ ]:
async def run_all_guardrails(answer, question, judge_options):
    """Run BOTH guardrail layers on an answer; collect every violation."""
    violations = []

    # Layer 1: the deterministic checks from guardrails.py
    det = run_deterministic_guardrails(answer)
    violations.extend(det.violations)

    # Layer 1b: the extra check you wrote in TODO 1
    violations.extend(check_has_numbers(answer))

    # Layer 2: the LLM judge
    judge_prompt = f"QUESTION:\n{question}\n\nANSWER:\n{answer}"
    judge_run = await run_agent("JudgeAgent", judge_prompt, judge_options)
    judge_approved, judge_reason = parse_verdict(judge_run.answer)
    if not judge_approved:
        violations.append(f"LLM judge rejected the answer: {judge_reason}")

    return {
        "passed": len(violations) == 0,
        "violations": violations,
        "judge_run": judge_run,
    }

print("[OK] run_all_guardrails defined")

## 8. Run a real answer through the guardrails

**TODO 3**: write the question — top 5 countries by revenue in 2011. A clean answer should pass every layer.

In [ ]:
# ----- TODO 3: write your question -----
QUESTION = "What were the top 5 countries by revenue in 2011?"
# ---------------------------------------

trace = Trace(model=DEV_MODEL, question=QUESTION)
with Timer() as t:
    run = await run_agent("DataAgent", QUESTION, data_options)
    guard = await run_all_guardrails(run.answer, QUESTION, judge_options)

judge_run = guard["judge_run"]
trace.input_tokens        = run.input_tokens + judge_run.input_tokens
trace.output_tokens       = run.output_tokens + judge_run.output_tokens
trace.cached_input_tokens = run.cached_input_tokens + judge_run.cached_input_tokens
trace.n_tool_calls        = run.n_tool_calls
trace.n_delegations       = 2     # the DataAgent + the LLM judge
trace.latency_ms          = t.elapsed_ms
trace.answer              = run.answer
trace.compute_cost()

print("----- Agent answer -----")
print(run.answer)
print()
print(f"Guardrails passed: {guard['passed']}")
print(f"Violations: {guard['violations']}")

## 9. Verify (acceptance criteria)

In [ ]:
# ----- TODO 4: write the assertion -----
missing = []
for country in expected_countries:
    if country not in run.answer:
        missing.append(country)
# ---------------------------------------

trace.passed = (missing == [] and guard["passed"])
print(f"Missing countries: {missing}")
print(f"Guardrails passed: {guard['passed']}")
print(f"Passed: {trace.passed}")
assert missing == [], f"Answer missed: {missing}"
assert guard["passed"], f"Guardrails flagged the real answer: {guard['violations']}"

# Phase 8 acceptance: prove the guardrails actually catch a known-bad answer
bad_check = run_deterministic_guardrails("Our Platinum customers earned 9,999,999,999.")
assert not bad_check.passed, "Guardrails should have caught the bad answer"

trace.append_jsonl()
print("\n[OK] Acceptance criteria met - good answer passed, bad answer caught, trace saved")

## 10. Quiz (~5 min — answer in a new markdown cell)

**TODO 5**: Answer in your own words.

1. **Two layers**: deterministic checks and an LLM judge catch *different* kinds of mistake. Give one example of an error the deterministic checks catch but the judge might miss — and one the judge catches but the rules can't.

   - **Rules catch, judge misses**: a number like `9,999,999,999` for monthly revenue. The deterministic check trips instantly on `max_value > 100M`; an LLM judge that doesn't know our dataset's order of magnitude might wave it through because the *sentence* reads plausibly.
   - **Judge catches, rules miss**: a question-dodge — "Q: top 5 countries by revenue in 2011? A: Here's a chart of customer segments by country." Every word is benign, every number is valid, no rule fires — but the judge spots that the answer is non-responsive.

2. **Hallucinated labels**: `check_no_hallucinated_segments` flags "Platinum", "Gold", etc. Why is a made-up segment name dangerous in a business report? Why can a plain rule catch this but not, say, a wrong revenue number?

   Downstream consumers act on label names: a slide deck says "we should target Platinum customers" and a stakeholder asks marketing to build a "Platinum" campaign — chasing a category that doesn't exist in our system. A rule catches it because the universe of valid labels is **finite and known** (`VIP, Regular, At-Risk, Churned`), so set-membership is decisive. Revenue numbers are continuous and unbounded — there's no static list of "valid revenues" to check against, so a rule can only catch wild outliers, not subtly-wrong figures.

3. **Critic vs guardrail**: Phase 2 had a Critic; Phase 8 has guardrails. They feel similar. What's the difference in *when* and *how* each one runs?

   The **Critic** is part of the *agent loop* — it reviews the worker mid-orchestration and can trigger a **revision**: critic rejects → worker tries again with the feedback. **Guardrails** are a *final gate* run after the orchestrator has produced an answer — they either let it through to the user or block/flag it (no revision loop by default). Critic = collaborator inside the team; guardrails = QA at the loading dock.

4. **Defence in depth**: why run several independent checks instead of one big one? What's the risk of relying on the LLM judge alone?

   Independent checks fail independently — a structural rule and an LLM judge have very different failure modes, so when one is fooled the other usually isn't. Relying on the judge alone means a single point of failure: it can be biased by clever wording, can hallucinate a "looks fine" verdict, or can drift across model updates. Cheap deterministic rules plus a semantic judge gives you a wider net at low marginal cost.

5. **Cost**: the LLM-judge adds an extra LLM call to every answer. When is that worth it? When might you run only the (free, instant) deterministic checks and skip the judge?

   Worth it on anything user-facing where a wrong answer has reputation or business cost — exec reports, customer-facing replies, anything that gets quoted. Skip the judge on internal high-volume pipelines (eval runs, batch backfills, ETL traces) where you've already validated quality and just want a cheap safety net against the egregious failures the rules already cover.

## Phase 8 done when:
- [x] TODO 1 (check_has_numbers) filled in
- [x] TODO 2 (LLM-judge system prompt) filled in
- [x] TODO 3 (your question) filled in
- [x] TODO 4 (assertion) filled in
- [x] TODO 5 (quiz) answered
- [x] Cell 9 shows the deterministic guardrails catching a bad answer
- [x] Cell 17 shows a real answer passing all guardrail layers
- [x] Notebook runs top-to-bottom without errors
- [x] `traces/traces.jsonl` has a new line

Then ping me — we'll review, then move to Phase 9 (evals & observability).